In [ ]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

# Middleware is used to summarize the conversation history when it exceeds a certain number of tokens. 
# This helps to keep the context manageable for the agent and prevents it from losing track of the conversation.
# SummarizationMiddleware is configured to trigger when the conversation exceeds 100 tokens and will keep the last message in the conversation history.
# Custom prompt can be added to the SummarizationMiddleware to customize the summarization behavior, but in this case, we are using the default behavior.

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [ ]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 1,000,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

In [ ]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

The user's primary goal is to gather information about Lunapolis, including its capital, weather conditions, population details, and socio-political climate.

## SUMMARY

- The capital of the moon is Lunapolis.
- The weather in Lunapolis features clear skies, with a high temperature of 120°C and a low of -100°C.
- Lunapolis has a population of 100,000 cheese miners.
- It is anticipated that the cheese miners' union may go on strike due to dissatisfaction with the new president.

## ARTIFACTS

None

## NEXT STEPS

None


## Trim/delete messages

In [ ]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

# Trim messages middleware is used to remove all the tool messages from the state before the agent processes the next message.
# Use before and after decorators to define the middleware function. 
# The before decorator is used to specify that the function should be called before the agent processes the next message.
# The after decorator is used to specify that the function should be called after the agent processes the next message.
# The trim_messages function takes the agent state and runtime as input and returns a dictionary with the messages to be removed from the state.

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [ ]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=120F voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='7b22bd81-f2a6-414d-99e0-771db7c95692'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='6d7e58ea-cf72-4b32-a35c-e7cbf4a8848e', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='4f326f52-f4dd-4699-a688-a6081a8b6191'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='c463c2db-1b81-4fd0-af51-aaf67be9ff60', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='464395ef-63ee-4394-8131-390c97fd2615'),
              AIMessage(content='I can’t read the device’s internal temperature from here. If i

In [ ]:
print(response["messages"][-1].content)

I can’t read the device’s internal temperature from here. If it’s not turning on, you can only roughly tell by touching the outside: it should be at or near room temperature. If the exterior feels hot to the touch (roughly 40°C/104°F or higher), that could indicate an issue and you should unplug and let it cool before trying again.

If you want to troubleshoot not turning on, here are quick steps (general):

- Try a different charger and outlet. Use the original charger if possible.
- Disconnect all accessories. If the device has a removable battery, reseat it or try powering with the charger alone.
- Do a soft reset: hold the power button for 10–20 seconds, then release and try again.
- Check for any signs of life: LEDs, sounds, or vibrations when you plug in or press the power button.
- If it still won’t power on, note the model and OS, and any indicator codes or beeps, and contact support.

If you share the exact device type (laptop, phone, tablet, desktop, etc.) and model, I can gi